In [27]:
# get the ml-10M100K folder path from s3 bucket
path = 's3://jonesh-test/ml-10M100K/'

# read ratings.dat from the s3 bucket location with delimeter '::' to Spark DataFrame with header
# inferSchema is set to true to convert any numerical values into integer/float
ratings = (spark.read
          .option('delimiter', '::')
          .option('inferSchema', 'true')
          .csv(path + 'ratings.dat')
          .toDF('UserID', 'MovieID', 'Rating', 'Timestamp'))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [28]:
# show top 5 rows from ratings DataFrame
ratings.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|    122|   5.0|838985046|
|     1|    185|   5.0|838983525|
|     1|    231|   5.0|838983392|
|     1|    292|   5.0|838983421|
|     1|    316|   5.0|838983392|
+------+-------+------+---------+
only showing top 5 rows

In [29]:
# read movies.dat from the s3 bucket location with delimeter '::' to Spark DataFrame with header
# inferSchema is set to true to convert any numerical values into integer/float
movies = (spark.read
         .option('delimiter', '::')
         .option('inferSchema', 'true')
         .csv(path + 'movies.dat')
         .toDF('MovieID', 'Title', 'Genres'))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [30]:
# show top 5 rows from movies DataFrame
movies.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows

In [31]:
# import all necessary functions like col, agg, sqrt, etc. from pyspark
from pyspark.sql.functions import *

# compute movie pairs rated by the same user for cosine similarity computation
# ratings dataframe is joined with itself based on same UserID and 
# r1.MovieID < r2.MovieID to ensure each movie pair appears only once (avoiding duplicates like [1,2] and [2,1])
# then used select to create four columns dataframe with MovieIDs and Ratings named movie_pairs
movie_pairs = ratings.alias('r1').join(ratings.alias('r2'),
                                       (col('r1.UserID') == col('r2.UserID')) &
                                       (col('r1.MovieID') < col('r2.MovieID'))
                                      ).select(col('r1.MovieID').alias('movie1'),
                                               col('r2.MovieID').alias('movie2'),
                                               col('r1.Rating').alias('rating1'),
                                               col('r2.Rating').alias('rating2'))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [32]:
# show top 5 rows from movie_pairs DataFrame
movie_pairs.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+------+-------+-------+
|movie1|movie2|rating1|rating2|
+------+------+-------+-------+
|     5|   253|    3.0|    3.0|
|     5|   345|    3.0|    4.0|
|     5|   454|    3.0|    3.0|
|     5|   736|    3.0|    4.0|
|     5|   780|    3.0|    5.0|
+------+------+-------+-------+
only showing top 5 rows

In [33]:
# using the movie_pairs to compute components needed for cosine similarity
# first grouped by movie1 and movie2 and count the number of users that rated both movies as numPairs
# then compute sum of element-wise products of the two rating vectors (sum_xy),
# sum of squared ratings for movie1 (sum_xx)
# sum of squared ratings for movie2 (sum_yy)
pair_scores = movie_pairs.groupBy('movie1', 'movie2').agg(
    count('*').alias('numPairs'),
    sum(col('rating1') * col('rating2')).alias('sum_xy'),
    sum(col('rating1') * col('rating1')).alias('sum_xx'),
    sum(col('rating2') * col('rating2')).alias('sum_yy'))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [34]:
# show top 5 rows from pair_scores DataFrame
pair_scores.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+------+--------+--------+--------+--------+
|movie1|movie2|numPairs|  sum_xy|  sum_xx|  sum_yy|
+------+------+--------+--------+--------+--------+
|     1|  2004|    1772|18875.25|28299.75|14830.75|
|     1|  2668|     588|  6289.5| 10397.5| 4421.25|
|     1|  2994|      33|   444.0|  485.75|  441.75|
|     1|  3809|    2208|29898.25| 36551.0|27035.25|
|     1|  4241|     199|  1160.0|  3125.0|  715.25|
+------+------+--------+--------+--------+--------+
only showing top 5 rows

In [35]:
# compute cosine similarity
cosine_similarities = pair_scores.withColumn('score', col('sum_xy') / (sqrt(col('sum_xx')) * sqrt(col('sum_yy'))))
# filter the cosine similarities where numPairs >= 10 i.e. only if more than 10 users have rated both movies
# and cosine similarity scores are greater than 0.95 to store only the most similar movie pairs
similarities_filtered = cosine_similarities.filter(col('numPairs') >= 10).filter(col('score') >= 0.95)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [36]:
# show top 5 rows from similarities_filtered DataFrame
similarities_filtered.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+------+--------+--------+--------+---------+------------------+
|movie1|movie2|numPairs|  sum_xy|  sum_xx|   sum_yy|             score|
+------+------+--------+--------+--------+---------+------------------+
|     1|  1208|    6449|102072.0|100414.0|113210.75|0.9573387913508588|
|     1|  1348|    1136|17080.75|17666.25| 18250.25|0.9512624225282171|
|     1|  1357|    2873| 44755.5|47380.25| 45733.25| 0.961461075725028|
|     1|  3304|     109| 1549.25| 1758.75|  1483.75|0.9590452167312737|
|     1|  3668|    1022|14785.75|16644.75|  14431.5|0.9540013377015786|
+------+------+--------+--------+--------+---------+------------------+
only showing top 5 rows

In [37]:
# get top 10 movies most similar to Toy Story (MovieID = 1)
# filter for pairs where movie1 = 1, then order by similarity score descending and limit to 10 results
toy_story_similarities = similarities_filtered.filter(col('movie1') == 1).orderBy(col('score').desc()).limit(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [38]:
# show top 5 rows from toy_story_similarities DataFrame
toy_story_similarities.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+------+--------+------+------+------+------------------+
|movie1|movie2|numPairs|sum_xy|sum_xx|sum_yy|             score|
+------+------+--------+------+------+------+------------------+
|     1| 36553|      10| 143.0|162.75| 126.5|0.9966215554619151|
|     1|  6336|      13| 180.5|207.75| 158.5|0.9946998598171956|
|     1| 41831|      17| 244.0|274.75|219.75|0.9930166580182394|
|     1| 60341|      10|137.25|170.75| 112.0|0.9924827921220798|
|     1|  8422|      17|234.25| 249.5| 223.5|0.9919860408658672|
|     1| 55895|      15| 227.5| 255.0| 207.0|0.9902073326897796|
|     1|  7273|      11| 157.5| 234.5| 108.0|0.9896856126772272|
|     1| 25931|      12|169.25| 217.5| 134.5|0.9895503677685183|
|     1| 32781|      24|332.25|439.75| 256.5| 0.989278067592776|
|     1| 62801|      17|266.75|267.25|272.25|0.9889210628340293|
+------+------+--------+------+------+------+------------------+

In [39]:
# formatting for final output
# join the movies dataframe with the toy_story_similarities based on the movieID for both movie1 and movie2
# select the Title from movies based on the join and add columns to top_similar_movies_with_names dataframe and scores
top_similar_movies_with_names = toy_story_similarities.join(
    movies.alias('m1'), toy_story_similarities.movie1 == col('m1.MovieID')).join(
    movies.alias('m2'), toy_story_similarities.movie2 == col('m2.MovieID')).select(
    col('m1.Title').alias('Movie Name'),
    col('m2.Title').alias('Similar Movies'),
    col('score'))
# order by score descending
top_similar_movies_with_names = top_similar_movies_with_names.orderBy(col('score').desc())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [40]:
# final result shows three columns with these exact names (Movie Name, Similar Movies, score)
top_similar_movies_with_names.select('Movie Name', 'Similar Movies', 'score').show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+--------------------+------------------+
|      Movie Name|      Similar Movies|             score|
+----------------+--------------------+------------------+
|Toy Story (1995)|In Old Chicago (1...|0.9966215554619151|
|Toy Story (1995)|Marooned in Iraq ...|0.9946998598171956|
|Toy Story (1995)|They Died with Th...|0.9930166580182394|
|Toy Story (1995)|Standard Operatin...|0.9924827921220798|
|Toy Story (1995)|    Kings Row (1942)|0.9919860408658672|
|Toy Story (1995)|Desperate Hours, ...|0.9902073326897796|
|Toy Story (1995)|Piece of the Acti...|0.9896856126772272|
|Toy Story (1995)|  Road to Rio (1947)|0.9895503677685183|
|Toy Story (1995)|       Hawaii (1966)| 0.989278067592776|
|Toy Story (1995)|Lone Wolf and Cub...|0.9889210628340293|
+----------------+--------------------+------------------+